# Sentinel-2 GeoTIFF Creator — Raw Bands and / or Computed Indices

This notebook produces GeoTIFF files from Sentinel-2 Level-2A imagery for a
user-defined area of interest.  It supports **two output formats**, in any
combination:

* **All-Bands GeoTIFF** — all 11 raw spectral bands (B01–B12, including B8A)
  stored as `uint16`.  Maximum information; downstream users can derive any
  index they need.
* **Indices GeoTIFF** — pre-computed NDVI, NDBI, NDWI as `float64`.  Smaller
  and ready for direct use in models that only need these three indices.

For each bounding box (MBR / TBR), each output format can be produced from:

* a **single best scene** (manually chosen low-cloud date), and / or
* a **median composite** across all scenes in the time window (cloud-filtered).

The configuration cell near the top of the notebook lets you toggle each axis
on or off so you only pay the runtime cost for outputs you actually need.

---

### Runtime guidance

The dominant cost in this notebook is the **median composite**: computing
`data.median(dim="time").compute()` triggers Dask to download every scene in
the time window for the requested bounding box.  For a city-scale AOI with
10–20 scenes this typically takes 5–15 minutes per bounding box.  By contrast,
the single-scene path only fetches one scene's worth of data, and choosing
between raw-band vs computed-index output adds only seconds.

The flags in the configuration cell are arranged so that the most expensive
toggles are obvious.

In [ ]:
# ---------------------------------------------------------------------------
# Clone the bc-II repository (contains bounding-box / MBR / TBR definitions
# and the .env_bin.txt configuration file).
# NOTE: This repo is for coordinate definitions ONLY — generated GeoTIFFs are
#       saved directly to Google Drive (mounted below).
# ---------------------------------------------------------------------------
!git clone https://github.com/chase-kusterer/bc-II.git

# Change the working directory into the cloned repo so we can access its files
%cd bc-II

# Verify the environment configuration file exists (suppress its output)
!cat .env_bin.txt > /dev/null

In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive so it appears as a local filesystem path in Colab.
# You will be prompted to authorise access the first time.
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# Define the output directory on Google Drive.
# This creates the folder if it doesn't already exist.
# TODO: Adjust the folder path to match your Drive structure.
# ---------------------------------------------------------------------------
import os

output_dir = "/content/drive/MyDrive/Business Challenge II/GeoTIFFs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Suppress non-critical warnings to keep notebook output clean
import warnings
warnings.filterwarnings('ignore')

# Install / upgrade the geospatial stack used for satellite data access
!pip install --upgrade rioxarray stackstac pystac-client planetary_computer odc-stac

# ---------------------------------------------------------------------------
# Core GIS & numerical libraries
# ---------------------------------------------------------------------------
import numpy as np                           # array operations
import xarray as xr                          # labelled multi-dimensional arrays
import matplotlib.pyplot as plt              # plotting
import rioxarray as rio                      # xarray <-> rasterio bridge
import rasterio                              # low-level raster I/O (GeoTIFF)
from matplotlib.cm import RdYlGn, jet, RdBu  # colormaps for index visualisation

# ---------------------------------------------------------------------------
# Microsoft Planetary Computer / STAC libraries
# ---------------------------------------------------------------------------
import stackstac                             # STAC items -> xarray DataArray
import pystac_client                         # search STAC catalogues
import planetary_computer                    # sign URLs for Planetary Computer
from odc.stac import stac_load               # alternative STAC loader (ODC)

## ⚙️ Run Configuration

Set the flags below to control what this notebook produces.  Cells throughout
the notebook check these flags and skip work that isn't needed.

**The `RUN_MBR`, `RUN_TBR`, and `MAKE_MEDIAN` flags drive runtime cost.**
Leaving them all `True` means the longest possible run; turning unnecessary
ones off can cut runtime by half or more.

The `WRITE_*` flags only affect which files end up on disk and are essentially
free to toggle.

In [ ]:
# ===========================================================================
# RUN CONFIGURATION
# ===========================================================================

# ---- Bounding-box selection ------------------------------------------------
# Each bounding box triggers an independent STAC search and data load.
# Disabling one cuts that half of the runtime entirely.
RUN_MBR = True   # Process the Minimum Bounding Rectangle
RUN_TBR = True   # Process the Tolerance Bounding Rectangle (buffered MBR)

# ---- Temporal aggregation --------------------------------------------------
# MAKE_SINGLE picks one cloud-free scene; cheap.
# MAKE_MEDIAN computes a pixel-wise median across all scenes; EXPENSIVE.
MAKE_SINGLE = True   # Produce single-scene outputs
MAKE_MEDIAN = True   # Produce median-composite outputs (Dask-heavy)

# ---- Output-format selection -----------------------------------------------
# These toggles determine which GeoTIFF files get written for each
# (bounding box) x (single/median) combination above.
# Both can be True; you'll get two files per combination.
WRITE_RAW_BANDS = True   # Write all 11 raw Sentinel-2 bands as uint16
WRITE_INDICES   = True   # Write computed NDVI, NDBI, NDWI as float64

# ---- Sanity check ----------------------------------------------------------
# Warn the user if the configuration produces no output.
if not (RUN_MBR or RUN_TBR):
    raise ValueError("Both RUN_MBR and RUN_TBR are False — nothing to do.")
if not (MAKE_SINGLE or MAKE_MEDIAN):
    raise ValueError("Both MAKE_SINGLE and MAKE_MEDIAN are False — nothing to do.")
if not (WRITE_RAW_BANDS or WRITE_INDICES):
    raise ValueError("Both WRITE_RAW_BANDS and WRITE_INDICES are False — no files would be written.")

# ---- Echo the planned workload --------------------------------------------
n_files = (int(RUN_MBR) + int(RUN_TBR)) \
        * (int(MAKE_SINGLE) + int(MAKE_MEDIAN)) \
        * (int(WRITE_RAW_BANDS) + int(WRITE_INDICES))
print(f"Planned outputs: {n_files} GeoTIFF file(s)")
print(f"  Bounding boxes : MBR={RUN_MBR}, TBR={RUN_TBR}")
print(f"  Aggregation    : Single={MAKE_SINGLE}, Median={MAKE_MEDIAN}")
print(f"  Formats        : Raw bands={WRITE_RAW_BANDS}, Indices={WRITE_INDICES}")

In [ ]:
# ---------------------------------------------------------------------------
# Canonical band list — single source of truth for which bands are loaded
# from Planetary Computer and (when WRITE_RAW_BANDS=True) written to the
# All-Bands GeoTIFF.  Band 1 = BANDS[0], Band 2 = BANDS[1], etc.
# ---------------------------------------------------------------------------
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07",
         "B08", "B8A", "B11", "B12"]
N_BANDS = len(BANDS)
print(f"Loading {N_BANDS} Sentinel-2 bands: {BANDS}")

In [ ]:
# ---------------------------------------------------------------------------
# Define the satellite imagery search window.
# Format: "YYYY-MM-DD/YYYY-MM-DD"
# Wider windows give more scenes (better cloud-free options) but dates far
# from the ground-truth collection date may not represent ground conditions.
# TODO: Replace the placeholder with actual dates.
# ---------------------------------------------------------------------------
time_window = "yyyy-mm-dd/yyyy-mm-dd" 

## Discover and Load the Data

Define the area of interest using latitude / longitude coordinates from the
project CSV.  Two bounding boxes are computed:

* **MBR (Minimum Bounding Rectangle)** — tightly fits the data points.
* **TBR (Tolerance Bounding Rectangle)** — adds a buffer around the MBR so
  that edge pixels are not cut off during raster extraction.

In [ ]:
import pandas as pd

# Read the dataset that contains ground-truth temperature observations
# TODO: Replace "file_path" with the actual path to your CSV
df = pd.read_csv("/content/bc-II/Data/filename.csv")

# ---------------------------------------------------------------------------
# Minimum Bounding Rectangle (MBR) — smallest rectangle enclosing all points
# ---------------------------------------------------------------------------
min_lat = df['Latitude'].min()
max_lat = df['Latitude'].max()
min_lon = df['Longitude'].min()
max_lon = df['Longitude'].max()

mbr_lower_left  = (min_lat, min_lon)
mbr_upper_right = (max_lat, max_lon)

print("MBR lower-left:", mbr_lower_left, " | upper-right:", mbr_upper_right)

# ---------------------------------------------------------------------------
# Tolerance Bounding Rectangle (TBR) — MBR + buffer for edge-pixel context
# ---------------------------------------------------------------------------
buffer = 0.1  # degrees (~11 km at the equator)

tbr_lower_left  = (min_lat - buffer, min_lon - buffer)
tbr_upper_right = (max_lat + buffer, max_lon + buffer)

print("TBR lower-left:", tbr_lower_left, " | upper-right:", tbr_upper_right)

In [ ]:
# ---------------------------------------------------------------------------
# Reformat bounding boxes to (west, south, east, north) — the convention
# expected by STAC search and rasterio.
# ---------------------------------------------------------------------------
mbr_bounds = (mbr_lower_left[1], mbr_lower_left[0],
              mbr_upper_right[1], mbr_upper_right[0])

tbr_bounds = (tbr_lower_left[1], tbr_lower_left[0],
              tbr_upper_right[1], tbr_upper_right[0])

print("MBR bounds (W, S, E, N):", mbr_bounds)
print("TBR bounds (W, S, E, N):", tbr_bounds)

In [ ]:
# ---------------------------------------------------------------------------
# Spatial resolution settings
# Sentinel-2 native resolution for visible/NIR bands is 10 m.  Because we
# reproject to EPSG:4326 (degrees), we convert metres -> degrees using the
# approximation 1° latitude ≈ 111 320 m.
# ---------------------------------------------------------------------------
resolution = 10                # metres per pixel
scale = resolution / 111320.0  # approximate degrees per pixel at the equator

### Helper Functions

The two functions below encapsulate the raw-bands and indices GeoTIFF writes.
Defining them once here keeps the per-bounding-box sections short and ensures
MBR and TBR outputs are produced identically.

In [ ]:
# ===========================================================================
# Helper: write all 11 raw Sentinel-2 bands as a multi-band GeoTIFF (uint16)
# ===========================================================================
def write_raw_bands_geotiff(dataset, filepath, transform, width, height):
    """
    Write every band in `dataset` (one of: slice_mbr, median_mbr, slice_tbr,
    median_tbr) to a multi-band GeoTIFF at `filepath`.

    Bands are written in the order defined by the global `BANDS` list and
    cast to uint16 (Sentinel-2's native dtype) — no precision loss.
    Each band is tagged with its Sentinel-2 name so downstream readers can
    look up bands by name (src.descriptions) instead of just by index.
    """
    with rasterio.open(
        filepath, 'w', driver='GTiff',
        width=width, height=height,
        crs='epsg:4326', transform=transform,
        count=N_BANDS, compress='lzw', dtype='uint16'
    ) as dst:
        for i, band_name in enumerate(BANDS, start=1):
            band_data = dataset[band_name].astype('uint16').values
            dst.write(band_data, i)
            dst.set_band_description(i, band_name)
    print(f"  [raw bands] {filepath}")


# ===========================================================================
# Helper: write 3 computed indices (NDVI, NDBI, NDWI) as float64 GeoTIFF
# ===========================================================================
def write_indices_geotiff(ndvi, ndbi, ndwi, filepath, transform, width, height):
    """
    Write the three pre-computed indices to a 3-band GeoTIFF.
    Band 1 = NDVI, Band 2 = NDBI, Band 3 = NDWI.
    """
    with rasterio.open(
        filepath, 'w', driver='GTiff',
        width=width, height=height,
        crs='epsg:4326', transform=transform,
        count=3, compress='lzw', dtype='float64'
    ) as dst:
        dst.write(ndvi, 1); dst.set_band_description(1, "NDVI")
        dst.write(ndbi, 2); dst.set_band_description(2, "NDBI")
        dst.write(ndwi, 3); dst.set_band_description(3, "NDWI")
    print(f"  [indices ] {filepath}")


# ===========================================================================
# Helper: compute NDVI / NDBI / NDWI from a band-bearing dataset
# ===========================================================================
def compute_indices(dataset, cast_to_float=True):
    """
    Returns (NDVI, NDBI, NDWI) computed from the dataset's spectral bands.
    `cast_to_float=True` is needed for raw uint16 data to avoid integer
    overflow; for already-float data (e.g. median composites) it is harmless.
    """
    if cast_to_float:
        b03 = dataset.B03.astype('float64')
        b04 = dataset.B04.astype('float64')
        b08 = dataset.B08.astype('float64')
        b11 = dataset.B11.astype('float64')
    else:
        b03, b04, b08, b11 = dataset.B03, dataset.B04, dataset.B08, dataset.B11

    ndvi = (b08 - b04) / (b08 + b04)
    ndbi = (b11 - b08) / (b11 + b08)
    ndwi = (b03 - b08) / (b03 + b08)
    return ndvi, ndbi, ndwi

### Sentinel-2 Bands Summary

| GeoTIFF Band # | S-2 Band | Description            | Native Resolution |
|---------------:|:--------:|------------------------|-------------------|
|              1 | B01      | Coastal Aerosol        | 60 m              |
|              2 | B02      | Blue                   | 10 m              |
|              3 | B03      | Green                  | 10 m              |
|              4 | B04      | Red                    | 10 m              |
|              5 | B05      | Red Edge (704 nm)      | 20 m              |
|              6 | B06      | Red Edge (740 nm)      | 20 m              |
|              7 | B07      | Red Edge (780 nm)      | 20 m              |
|              8 | B08      | NIR (833 nm)           | 10 m              |
|              9 | B8A      | NIR narrow (864 nm)    | 20 m              |
|             10 | B11      | SWIR (1.6 µm)          | 20 m              |
|             11 | B12      | SWIR (2.2 µm)          | 20 m              |

The All-Bands GeoTIFF preserves all 11 bands; the Indices GeoTIFF stores
NDVI / NDBI / NDWI derived from B03, B04, B08, and B11.

### Spectral Indices Reference

For users selecting `WRITE_INDICES = True`, the three indices written are:

* **NDVI = (B08 − B04) / (B08 + B04)** — vegetation greenness.
  *< 0.25*: bare/urban/water · *0.25–0.6*: grass/crops · *0.6–1.0*: dense
  vegetation.
* **NDBI = (B11 − B08) / (B11 + B08)** — built-up density.
  *< 0*: vegetation/water · *> 0*: built-up surfaces.
* **NDWI = (B03 − B08) / (B03 + B08)** — surface water.
  *> 0*: water · *< 0*: non-water.

# MBR Image Extraction

The cells in this section run only when `RUN_MBR = True` in the configuration
cell.  When `RUN_MBR = False`, every cell below short-circuits and prints a
skip notice — no Planetary Computer calls or Dask work is triggered.

In [ ]:
# ---------------------------------------------------------------------------
# MBR: STAC search and data load.
# Wrapped in `if RUN_MBR` so the entire MBR pipeline can be skipped.
# ---------------------------------------------------------------------------
if RUN_MBR:
    # Connect to Microsoft Planetary Computer's STAC API
    stac = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1"
    )

    # Search for Sentinel-2 L2A scenes intersecting the MBR with <30% clouds
    search_mbr = stac.search(
        bbox=mbr_bounds,
        datetime=time_window,
        collections=["sentinel-2-l2a"],
        query={"eo:cloud_cover": {"lt": 30}},
    )
    items_mbr = list(search_mbr.get_items())
    print(f"MBR: {len(items_mbr)} scenes found within the time window.")
else:
    print("MBR: skipped (RUN_MBR=False)")

In [ ]:
# ---------------------------------------------------------------------------
# Sign each MBR STAC item with a short-lived SAS token for download access
# ---------------------------------------------------------------------------
if RUN_MBR:
    signed_items_mbr = [planetary_computer.sign(item).to_dict() for item in items_mbr]

In [ ]:
# ---------------------------------------------------------------------------
# Load MBR satellite data via odc.stac.stac_load.
# Lazy operation — actual download deferred until .compute() or plotting.
# ---------------------------------------------------------------------------
if RUN_MBR:
    data_mbr = stac_load(
        items_mbr,
        bands=BANDS,
        crs="EPSG:4326",
        resolution=scale,
        chunks={"x": 2048, "y": 2048},
        dtype="uint16",
        patch_url=planetary_computer.sign,
        bbox=mbr_bounds,
    )
    display(data_mbr)

### MBR — RGB Time-Series Preview

Inspect every scene returned by the search to identify a low-cloud date for
the single-scene output.  Scene numbering starts at 0 (top-left) and proceeds
left-to-right, row-by-row.

In [ ]:
# ---------------------------------------------------------------------------
# Plot all MBR scenes as small-multiple RGB thumbnails
# (Useful for picking a single-scene date AND for sanity-checking median input)
# ---------------------------------------------------------------------------
if RUN_MBR:
    plot_data_mbr = data_mbr[["B04", "B03", "B02"]].to_array()
    plot_data_mbr.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
    plt.show()

### MBR — Single-Scene Path

Runs only when both `RUN_MBR` and `MAKE_SINGLE` are `True`.

After viewing the thumbnails above, update the `time=` index below to the
scene with the least cloud cover and best spatial completeness.

In [ ]:
# ---------------------------------------------------------------------------
# Display the chosen single MBR scene at full size
# TODO: Update time index and the title's date string
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_SINGLE:
    SINGLE_TIME_INDEX_MBR = 9   # <-- adjust to your chosen scene index
    fig, ax = plt.subplots(figsize=(6, 6))
    plot_data_mbr.isel(time=SINGLE_TIME_INDEX_MBR).plot.imshow(
        robust=True, ax=ax, vmin=0, vmax=2500
    )
    ax.set_title(f"MBR RGB Single Date (time={SINGLE_TIME_INDEX_MBR}): <UPDATE DATE>")
    ax.axis('off')
    plt.show()

    # Extract the chosen time slice for downstream use
    slice_mbr = data_mbr.isel(time=SINGLE_TIME_INDEX_MBR)

In [ ]:
# ---------------------------------------------------------------------------
# Compute single-scene indices.  These are needed both for previews AND for
# the indices GeoTIFF (when WRITE_INDICES=True).  Cheap; computed from data
# already in memory.
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_SINGLE:
    ndvi_slice_mbr, ndbi_slice_mbr, ndwi_slice_mbr = compute_indices(
        slice_mbr, cast_to_float=True
    )

In [ ]:
# ---------------------------------------------------------------------------
# Preview MBR single-scene indices side-by-side
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_SINGLE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ndvi_slice_mbr.plot.imshow(ax=axes[0], vmin=0.0, vmax=0.8, cmap="RdYlGn")
    axes[0].set_title("MBR Single NDVI"); axes[0].axis('off')
    ndbi_slice_mbr.plot.imshow(ax=axes[1], vmin=-0.1, vmax=0.04, cmap="jet")
    axes[1].set_title("MBR Single NDBI"); axes[1].axis('off')
    ndwi_slice_mbr.plot.imshow(ax=axes[2], vmin=-0.4, vmax=0.2, cmap="RdBu")
    axes[2].set_title("MBR Single NDWI"); axes[2].axis('off')
    plt.tight_layout(); plt.show()

### MBR — Median Composite Path  ⚠️ EXPENSIVE

Runs only when both `RUN_MBR` and `MAKE_MEDIAN` are `True`.

`data.median(dim="time").compute()` is the **most expensive operation in the
notebook** — it forces Dask to download and process every scene in the time
window for the MBR.  Skip this section if you only need single-scene outputs.

In [ ]:
# ---------------------------------------------------------------------------
# Compute the MBR median composite.
# This is the dominant cost in the notebook (5-15 min for typical city AOIs).
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_MEDIAN:
    print("Computing MBR median composite — this may take several minutes...")
    median_mbr = data_mbr.median(dim="time").compute()
    print("Done.")

In [ ]:
# ---------------------------------------------------------------------------
# Plot the cloud-free MBR median RGB composite
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_MEDIAN:
    fig, ax = plt.subplots(figsize=(6, 6))
    median_mbr[["B04", "B03", "B02"]].to_array().plot.imshow(
        robust=True, ax=ax, vmin=0, vmax=2500
    )
    ax.set_title("MBR RGB Median Composite"); ax.axis('off')
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compute median-composite indices.
# `cast_to_float=False` because the median operation already produced floats.
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_MEDIAN:
    ndvi_median_mbr, ndbi_median_mbr, ndwi_median_mbr = compute_indices(
        median_mbr, cast_to_float=False
    )

In [ ]:
# ---------------------------------------------------------------------------
# Preview MBR median indices side-by-side
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_MEDIAN:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ndvi_median_mbr.plot.imshow(ax=axes[0], vmin=0.0, vmax=0.8, cmap="RdYlGn")
    axes[0].set_title("MBR Median NDVI"); axes[0].axis('off')
    ndbi_median_mbr.plot.imshow(ax=axes[1], vmin=-0.1, vmax=0.04, cmap="jet")
    axes[1].set_title("MBR Median NDBI"); axes[1].axis('off')
    ndwi_median_mbr.plot.imshow(ax=axes[2], vmin=-0.4, vmax=0.2, cmap="RdBu")
    axes[2].set_title("MBR Median NDWI"); axes[2].axis('off')
    plt.tight_layout(); plt.show()

### Save MBR GeoTIFF Files

Up to four files are produced for MBR depending on the configuration flags:

| File suffix                    | Requires                              |
|--------------------------------|---------------------------------------|
| `_Single_AllBands.tiff`        | `MAKE_SINGLE` & `WRITE_RAW_BANDS`     |
| `_Single_Indices.tiff`         | `MAKE_SINGLE` & `WRITE_INDICES`       |
| `_Median_AllBands.tiff`        | `MAKE_MEDIAN` & `WRITE_RAW_BANDS`     |
| `_Median_Indices.tiff`         | `MAKE_MEDIAN` & `WRITE_INDICES`       |

**TODO:** Update the filename prefix below with your city name and dates.

In [ ]:
# ---------------------------------------------------------------------------
# MBR output filename prefix and full Drive paths.
# All four possible MBR filenames share this prefix; suffix encodes the
# (single/median) x (raw/indices) combination.
# TODO: Replace placeholders.
# ---------------------------------------------------------------------------
if RUN_MBR:
    prefix_mbr = "City_MBR(yyyy-mm-dd)"   # <-- update with city + date(s)

    fp_mbr_single_raw     = os.path.join(output_dir, f"{prefix_mbr}_Single_AllBands.tiff")
    fp_mbr_single_indices = os.path.join(output_dir, f"{prefix_mbr}_Single_Indices.tiff")
    fp_mbr_median_raw     = os.path.join(output_dir, f"{prefix_mbr}_Median_AllBands.tiff")
    fp_mbr_median_indices = os.path.join(output_dir, f"{prefix_mbr}_Median_Indices.tiff")

In [ ]:
# ---------------------------------------------------------------------------
# Build affine transforms and attach CRS metadata for the MBR datasets.
# Done conditionally so we don't reference uncomputed slice/median objects.
# ---------------------------------------------------------------------------
if RUN_MBR and MAKE_SINGLE:
    height_slice_mbr = slice_mbr.dims["latitude"]
    width_slice_mbr  = slice_mbr.dims["longitude"]
    gt_slice_mbr = rasterio.transform.from_bounds(
        mbr_lower_left[1], mbr_lower_left[0],
        mbr_upper_right[1], mbr_upper_right[0],
        width_slice_mbr, height_slice_mbr
    )
    slice_mbr.rio.write_crs("epsg:4326", inplace=True)
    slice_mbr.rio.write_transform(transform=gt_slice_mbr, inplace=True)

if RUN_MBR and MAKE_MEDIAN:
    height_median_mbr = median_mbr.dims["latitude"]
    width_median_mbr  = median_mbr.dims["longitude"]
    gt_median_mbr = rasterio.transform.from_bounds(
        mbr_lower_left[1], mbr_lower_left[0],
        mbr_upper_right[1], mbr_upper_right[0],
        width_median_mbr, height_median_mbr
    )
    median_mbr.rio.write_crs("epsg:4326", inplace=True)
    median_mbr.rio.write_transform(transform=gt_median_mbr, inplace=True)

In [ ]:
# ---------------------------------------------------------------------------
# Write MBR GeoTIFFs.  Each of the four possible files is gated by its own
# combination of flags so only the requested files are produced.
# ---------------------------------------------------------------------------
if RUN_MBR:
    print("Writing MBR GeoTIFFs:")

    if MAKE_SINGLE and WRITE_RAW_BANDS:
        write_raw_bands_geotiff(
            slice_mbr, fp_mbr_single_raw,
            gt_slice_mbr, width_slice_mbr, height_slice_mbr
        )

    if MAKE_SINGLE and WRITE_INDICES:
        write_indices_geotiff(
            ndvi_slice_mbr, ndbi_slice_mbr, ndwi_slice_mbr,
            fp_mbr_single_indices,
            gt_slice_mbr, width_slice_mbr, height_slice_mbr
        )

    if MAKE_MEDIAN and WRITE_RAW_BANDS:
        write_raw_bands_geotiff(
            median_mbr, fp_mbr_median_raw,
            gt_median_mbr, width_median_mbr, height_median_mbr
        )

    if MAKE_MEDIAN and WRITE_INDICES:
        write_indices_geotiff(
            ndvi_median_mbr, ndbi_median_mbr, ndwi_median_mbr,
            fp_mbr_median_indices,
            gt_median_mbr, width_median_mbr, height_median_mbr
        )

# TBR Image Extraction

The cells in this section run only when `RUN_TBR = True`.  The structure
mirrors the MBR section above; comments are kept brief to avoid repetition.

In [ ]:
# TBR: STAC search and item materialisation
if RUN_TBR:
    stac = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1"
    )
    search_tbr = stac.search(
        bbox=tbr_bounds,
        datetime=time_window,
        collections=["sentinel-2-l2a"],
        query={"eo:cloud_cover": {"lt": 30}},
    )
    items_tbr = list(search_tbr.get_items())
    print(f"TBR: {len(items_tbr)} scenes found within the time window.")
else:
    print("TBR: skipped (RUN_TBR=False)")

In [ ]:
# Sign TBR STAC items
if RUN_TBR:
    signed_items_tbr = [planetary_computer.sign(item).to_dict() for item in items_tbr]

In [ ]:
# Load TBR satellite data (lazy)
if RUN_TBR:
    data_tbr = stac_load(
        items_tbr,
        bands=BANDS,
        crs="EPSG:4326",
        resolution=scale,
        chunks={"x": 2048, "y": 2048},
        dtype="uint16",
        patch_url=planetary_computer.sign,
        bbox=tbr_bounds,
    )
    display(data_tbr)

### TBR — RGB Time-Series Preview

In [ ]:
# Plot all TBR scenes as small-multiple RGB thumbnails
if RUN_TBR:
    plot_data_tbr = data_tbr[["B04", "B03", "B02"]].to_array()
    plot_data_tbr.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
    plt.show()

### TBR — Single-Scene Path

Runs when `RUN_TBR` and `MAKE_SINGLE` are both `True`.  Update the time index
below.

In [ ]:
# Display chosen TBR single scene and extract the slice
if RUN_TBR and MAKE_SINGLE:
    SINGLE_TIME_INDEX_TBR = 9   # <-- adjust to your chosen scene index
    fig, ax = plt.subplots(figsize=(6, 6))
    plot_data_tbr.isel(time=SINGLE_TIME_INDEX_TBR).plot.imshow(
        robust=True, ax=ax, vmin=0, vmax=2500
    )
    ax.set_title(f"TBR RGB Single Date (time={SINGLE_TIME_INDEX_TBR}): <UPDATE DATE>")
    ax.axis('off')
    plt.show()
    slice_tbr = data_tbr.isel(time=SINGLE_TIME_INDEX_TBR)

In [ ]:
# Compute single-scene TBR indices (cheap, in-memory)
if RUN_TBR and MAKE_SINGLE:
    ndvi_slice_tbr, ndbi_slice_tbr, ndwi_slice_tbr = compute_indices(
        slice_tbr, cast_to_float=True
    )

In [ ]:
# Preview TBR single-scene indices side-by-side
if RUN_TBR and MAKE_SINGLE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ndvi_slice_tbr.plot.imshow(ax=axes[0], vmin=0.0, vmax=0.8, cmap="RdYlGn")
    axes[0].set_title("TBR Single NDVI"); axes[0].axis('off')
    ndbi_slice_tbr.plot.imshow(ax=axes[1], vmin=-0.1, vmax=0.04, cmap="jet")
    axes[1].set_title("TBR Single NDBI"); axes[1].axis('off')
    ndwi_slice_tbr.plot.imshow(ax=axes[2], vmin=-0.4, vmax=0.2, cmap="RdBu")
    axes[2].set_title("TBR Single NDWI"); axes[2].axis('off')
    plt.tight_layout(); plt.show()

### TBR — Median Composite Path  ⚠️ EXPENSIVE

Same Dask cost warning as the MBR median.

In [ ]:
# Compute TBR median composite (Dask-heavy)
if RUN_TBR and MAKE_MEDIAN:
    print("Computing TBR median composite — this may take several minutes...")
    median_tbr = data_tbr.median(dim="time").compute()
    print("Done.")

In [ ]:
# Plot TBR median RGB
if RUN_TBR and MAKE_MEDIAN:
    fig, ax = plt.subplots(figsize=(6, 6))
    median_tbr[["B04", "B03", "B02"]].to_array().plot.imshow(
        robust=True, ax=ax, vmin=0, vmax=2500
    )
    ax.set_title("TBR RGB Median Composite"); ax.axis('off')
    plt.show()

In [ ]:
# Compute TBR median indices (already-float source, so no cast needed)
if RUN_TBR and MAKE_MEDIAN:
    ndvi_median_tbr, ndbi_median_tbr, ndwi_median_tbr = compute_indices(
        median_tbr, cast_to_float=False
    )

In [ ]:
# Preview TBR median indices side-by-side
if RUN_TBR and MAKE_MEDIAN:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ndvi_median_tbr.plot.imshow(ax=axes[0], vmin=0.0, vmax=0.8, cmap="RdYlGn")
    axes[0].set_title("TBR Median NDVI"); axes[0].axis('off')
    ndbi_median_tbr.plot.imshow(ax=axes[1], vmin=-0.1, vmax=0.04, cmap="jet")
    axes[1].set_title("TBR Median NDBI"); axes[1].axis('off')
    ndwi_median_tbr.plot.imshow(ax=axes[2], vmin=-0.4, vmax=0.2, cmap="RdBu")
    axes[2].set_title("TBR Median NDWI"); axes[2].axis('off')
    plt.tight_layout(); plt.show()

### Save TBR GeoTIFF Files

Same four-file matrix as the MBR section.  **TODO:** Update the prefix.

In [ ]:
# TBR output filename prefix and full Drive paths
if RUN_TBR:
    prefix_tbr = "City_TBR(yyyy-mm-dd)"   # <-- update with city + date(s)

    fp_tbr_single_raw     = os.path.join(output_dir, f"{prefix_tbr}_Single_AllBands.tiff")
    fp_tbr_single_indices = os.path.join(output_dir, f"{prefix_tbr}_Single_Indices.tiff")
    fp_tbr_median_raw     = os.path.join(output_dir, f"{prefix_tbr}_Median_AllBands.tiff")
    fp_tbr_median_indices = os.path.join(output_dir, f"{prefix_tbr}_Median_Indices.tiff")

In [ ]:
# Build affine transforms and attach CRS metadata for TBR datasets
if RUN_TBR and MAKE_SINGLE:
    height_slice_tbr = slice_tbr.dims["latitude"]
    width_slice_tbr  = slice_tbr.dims["longitude"]
    gt_slice_tbr = rasterio.transform.from_bounds(
        tbr_lower_left[1], tbr_lower_left[0],
        tbr_upper_right[1], tbr_upper_right[0],
        width_slice_tbr, height_slice_tbr
    )
    slice_tbr.rio.write_crs("epsg:4326", inplace=True)
    slice_tbr.rio.write_transform(transform=gt_slice_tbr, inplace=True)

if RUN_TBR and MAKE_MEDIAN:
    height_median_tbr = median_tbr.dims["latitude"]
    width_median_tbr  = median_tbr.dims["longitude"]
    gt_median_tbr = rasterio.transform.from_bounds(
        tbr_lower_left[1], tbr_lower_left[0],
        tbr_upper_right[1], tbr_upper_right[0],
        width_median_tbr, height_median_tbr
    )
    median_tbr.rio.write_crs("epsg:4326", inplace=True)
    median_tbr.rio.write_transform(transform=gt_median_tbr, inplace=True)

In [ ]:
# Write TBR GeoTIFFs (gated by flag combinations, identical pattern to MBR)
if RUN_TBR:
    print("Writing TBR GeoTIFFs:")

    if MAKE_SINGLE and WRITE_RAW_BANDS:
        write_raw_bands_geotiff(
            slice_tbr, fp_tbr_single_raw,
            gt_slice_tbr, width_slice_tbr, height_slice_tbr
        )

    if MAKE_SINGLE and WRITE_INDICES:
        write_indices_geotiff(
            ndvi_slice_tbr, ndbi_slice_tbr, ndwi_slice_tbr,
            fp_tbr_single_indices,
            gt_slice_tbr, width_slice_tbr, height_slice_tbr
        )

    if MAKE_MEDIAN and WRITE_RAW_BANDS:
        write_raw_bands_geotiff(
            median_tbr, fp_tbr_median_raw,
            gt_median_tbr, width_median_tbr, height_median_tbr
        )

    if MAKE_MEDIAN and WRITE_INDICES:
        write_indices_geotiff(
            ndvi_median_tbr, ndbi_median_tbr, ndwi_median_tbr,
            fp_tbr_median_indices,
            gt_median_tbr, width_median_tbr, height_median_tbr
        )

## Final Summary — Files Written

The cell below lists every GeoTIFF that was actually produced (skipping any
combinations that were disabled in the configuration cell) with its size on
disk.

In [ ]:
# ---------------------------------------------------------------------------
# Build the list of files we expect to exist based on the flags, then check
# each one.  Reports size in MB for files that were written.
# ---------------------------------------------------------------------------
expected = []
if RUN_MBR:
    if MAKE_SINGLE and WRITE_RAW_BANDS: expected.append(fp_mbr_single_raw)
    if MAKE_SINGLE and WRITE_INDICES:   expected.append(fp_mbr_single_indices)
    if MAKE_MEDIAN and WRITE_RAW_BANDS: expected.append(fp_mbr_median_raw)
    if MAKE_MEDIAN and WRITE_INDICES:   expected.append(fp_mbr_median_indices)
if RUN_TBR:
    if MAKE_SINGLE and WRITE_RAW_BANDS: expected.append(fp_tbr_single_raw)
    if MAKE_SINGLE and WRITE_INDICES:   expected.append(fp_tbr_single_indices)
    if MAKE_MEDIAN and WRITE_RAW_BANDS: expected.append(fp_tbr_median_raw)
    if MAKE_MEDIAN and WRITE_INDICES:   expected.append(fp_tbr_median_indices)

print(f"Files written ({len(expected)}):")
for f in expected:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {size_mb:7.1f} MB  {f}")
    else:
        print(f"  MISSING       {f}")

---
### Reading the GeoTIFFs Back Later

Both file types open identically in `rasterio`; the difference is only the
band layout.

```python
import rasterio

# All-Bands file: 11 bands, uint16
with rasterio.open(".../City_MBR(...)_Single_AllBands.tiff") as src:
    print(src.descriptions)        # ('B01', 'B02', ..., 'B12')
    b08 = src.read(src.descriptions.index("B08") + 1)  # NIR

# Indices file: 3 bands, float64
with rasterio.open(".../City_MBR(...)_Single_Indices.tiff") as src:
    print(src.descriptions)        # ('NDVI', 'NDBI', 'NDWI')
    ndvi = src.read(1)
```